# Select-swap QROM

Blocked lookup loads one high-address block, routes the low-address cell to the first bank word, and XORs that word into the destination. The inverse routing and second load restore the bank. A borrowed bank can initially hold arbitrary coherent data. Its extra route–XOR–unroute sequence cancels the contribution of the borrowed value.

Adapted from the Qualtran v0.7.0 SelectSwapQROM tutorial. Copyright 2023 Google LLC. Licensed under Apache-2.0.

In [ ]:
from collections import Counter

import numpy as np
import torch
from IPython.display import SVG, Markdown, display
from sympy import symbols

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.resource_model import (
    estimate_component_resources,
)
from flagquantum.compiler.registers.lookup.qroam_resource_model import (
    source_select_swap_qrom_cost,
)
from flagquantum.compiler.registers.lookup.qrom_table import QromTable
from flagquantum.compiler.registers.lookup.select_swap_qrom import select_swap_qrom_ir
from flagquantum.compiler.registers.lookup.select_swap_qrom_calls import (
    SelectSwapRouteCall,
    select_swap_qrom_calls,
)
from flagquantum.compiler.registers.lookup.source_qroam_clean import QroamCleanTable
from flagquantum.compiler.registers.lookup.symbolic_qroam_resource_model import (
    source_blocked_lookup_resource_formula,
)
from flagquantum.drawer.call_graph import (
    export_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

## Example signatures

Both datasets share the same address and supply one scalar target word each. The concrete fixtures use one- or two-dimensional addresses. The shape-only examples declare one and two addressing axes for each bank policy. Bank words and temporary AND work are bound separately from these logical ports.

In [ ]:
line = QromTable.from_data(np.arange(5), np.arange(5) + 1)
grid_data = np.arange(25).reshape(5, 5)
grid = QromTable.from_data(grid_data, grid_data + 1)
n, m, b, b1, b2, k1, k2, control = symbols("N M b b1 b2 k1 k2 c", positive=True, integer=True)
one_axis = QromTable.from_shape((n,), (b,), num_controls=control)
two_axes = QromTable.from_shape((n, m), (b1, b2), num_controls=control)
assert tuple(line.register_layout) == (("selection", 3), ("target0_", 3), ("target1_", 3))
display(SVG(field_signature_svg((
    ("Two datasets", line.signature, {}),
    ("Two axes", grid.signature, {}),
    ("Borrowed, one axis", one_axis.signature, {}),
    ("Borrowed, two axes", two_axes.signature, {}),
    ("Clean, one axis", one_axis.signature, {}),
    ("Clean, two axes", two_axes.signature, {}),
))))

In [ ]:
formulas = {
    (dirty, len(table.shape)): source_blocked_lookup_resource_formula(
        table.shape, table.target_bitsizes, logs,
        control_count=table.num_controls, dirty=dirty,
    )
    for dirty in (False, True)
    for table, logs in ((one_axis, (k1,)), (two_axes, (k1, k2)))
}
formulas

## Original direct calls

The five-row example uses source default block sizes. A second graph with explicit blocks shows the two exchange directions and the extra borrowed-bank copy. The batched QROM is self-adjoint, so its two calls share a node.

In [ ]:
def title(node):
    if isinstance(node, QromTable):
        return f"QROM{node.shape} widths {node.target_bitsizes}"
    if isinstance(node, SelectSwapRouteCall):
        name = "SwapWithZero†" if node.adjoint else "SwapWithZero"
        return f"{name} {node.banks} width {node.width}"
    return str(node)

for label, logs in (("SelectSwapQROM (default)", None), ("SelectSwapQROM (block 2)", (1,))):
    declared = select_swap_qrom_calls(line, logs)
    graph = symbolic_call_graph(label, {label: declared}, max_depth=1)
    display(SVG(render_call_graph_svg(export_call_graph_dot(label, graph.direct_calls, title=title))))
    display(Markdown(format_call_graph_markdown(label, graph.direct_calls, graph.leaf_counts, title=title)))

## Append and verify bank restoration

One address has an enabled control and a complex amplitude. The other has a disabled control and a real amplitude. In the borrowed case their bank contents differ. The expected vector preserves both bank states and phases, with the target XOR applied only in the enabled branch. Work wires start and end at zero.

In [ ]:
table = QroamCleanTable((4,), (1,), ((0, 1, 1, 0),))
counts = {}
for dirty, enabled, disabled in ((False, 80, 40), (True, 82, 44)):
    initial = torch.zeros(128, dtype=torch.complex128)
    initial[enabled], initial[disabled] = 0.6j, 0.8
    expected = torch.zeros_like(initial)
    expected[enabled ^ 8], expected[disabled] = initial[enabled], initial[disabled]
    body = select_swap_qrom_ir(
        table, (0,), ((1, 2),), ((3,),), (((4,), (5,)),), (6,), (0,), dirty=dirty,
    )
    circuit = DynamicCircuit(7, inputs=initial, dtype=torch.complex128)
    circuit.append(FieldOperation(7, body))
    result = run_dynamic(circuit, shots=8, seed=41, strategy="trajectory")
    torch.testing.assert_close(result.final_states, expected.expand(8, -1), atol=1e-12, rtol=0)
    model = source_select_swap_qrom_cost(table, 1, (2,), (1,), dirty=dirty)
    counts[dirty] = {"emitted": dict(Counter(g.name for g in body)), "source_model": estimate_component_resources(model)}
counts